# Lv et al. 2025 Microstate Reproduction

Best-effort comparator. `paper_style` preserves paper comparability; `honest_grouped` uses grouped outer CV and training-subject-only feature selection. Group maps remain explicitly **transductive**, not fully inductive.

# 1. Setup

In [ ]:
import os, sys, json, glob, random, hashlib, builtins, platform, inspect
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import signal, stats
from sklearn.model_selection import StratifiedKFold, GroupKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.covariance import OAS
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix, f1_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | cwd: {Path.cwd()}")

# 2. Configuration
## 2.1 Domain Defaults
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # Paths / run identity
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-lv2025-microstate-reproduction"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "lv2025_microstate_reproduction_honest_grouped",
    "config_note": "Paper-style comparator plus grouped, nested feature selection; group maps are explicitly transductive.",
    # Dataset: exact mapped Lv cohort; QC flags but does not exclude by default.
    "subjects_to_use": [1,3,7,9,10,11,14,15,17,29,31,32,37,41], "exclude_subjects": [],
    "qc_max_mean_eeg_std_uv": 50.0, "exclude_qc_failures": False,
    # Preprocessing / microstates
    "sfreq": 500, "resample_sfreq": 250, "filter_band_hz": [1.0,30.0], "reference_mode": "average",
    "n_states": 4, "gfp_min_peak_distance_ms": 30.0, "gfp_max_peaks": 20000, "kmeans_n_init": 20,
    # Evaluation
    "evaluation_mode": "honest_grouped", # paper_style | honest_grouped
    "allow_transductive_group_maps": True, "group_cv": "leave_one_subject_out", "inner_group_folds": 5,
    "feature_mode": "fdr_then_exploratory_fallback", "fdr_alpha": 0.05,
    "paper_reported_features": ["A_duration","A_occurrence","A_coverage","C_duration","C_occurrence","C_coverage","B_to_A","D_to_A","D_to_C"],
    "paper_state_label_map": None, # e.g. {"MS0":"A",...}; exact comparator runs only when supplied
    "fallback_feature_count": 9, "classifiers": ["lda","svm","knn"], "knn_n_neighbors": 10,
    # Reproducibility
    "seed": 2026, "set_seed": True,
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _safe_write_text(stream, text):
    try: stream.write(text)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"
        stream.write(text.encode(enc, errors="replace").decode(enc, errors="replace"))
def _timestamped_print(*args, **kwargs):
    sep, end = kwargs.pop("sep", " "), kwargs.pop("end", "\n")
    flush, target = kwargs.pop("flush", False), kwargs.pop("file", None)
    message = sep.join(str(a) for a in args)
    stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}" if message else ""
    for stream in ([target] if target is not None else [sys.stdout, _LOG_FILE_HANDLE]): _safe_write_text(stream, stamped + end)
    if flush: _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f: json.dump(CONFIG, f, indent=2)
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
BASE_SEED = int(CONFIG["seed"])
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed); random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError: pass
if CONFIG["set_seed"]: seed_everything(BASE_SEED)
print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Data Loading Helpers

In [ ]:
EEG_IDX = [i for i in range(30) if i != 17]
CH_NAMES = ["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4","T3","T4","CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"]
def find_files(root):
    files = sorted(Path(root).glob("sub-*/sub-*_eeg.mat"))
    if not files: raise FileNotFoundError(root)
    return files
def load_subject(path):
    eeg = sio.loadmat(path)["eeg"][0, 0]
    raw = np.asarray(eeg["rawdata"], float); y = np.asarray(eeg["label"]).ravel().astype(int) - 1
    marker = raw[:, 32]; onsets=[]
    for m in marker:
        idx=np.flatnonzero(m == 2); valid=idx[(idx >= 800) & (idx <= 1300)]
        onsets.append(int(valid[0]) if len(valid) else 1003)
    sid=int(path.parent.name.split("-")[1])
    return sid, raw[:, EEG_IDX], y, np.asarray(onsets)
def selected_files():
    keep = CONFIG["subjects_to_use"]
    return [p for p in find_files(CONFIG["source_extract_dir"]) if keep is None or int(p.parent.name.split("-")[1]) in set(keep)]

## 3.2 Preprocessing and QC
QC flags are retained by default; `sub-15` is not silently removed. `reference_mode` is applied exactly.

In [ ]:
try:
    import mne
    from pycrostates.cluster import ModKMeans
    from pycrostates.preprocessing import extract_gfp_peaks
except ImportError as e: raise ImportError("Lv reproduction requires mne, pycrostates, and statsmodels") from e
def preprocess(raw, onsets):
    trials=[]
    for x,o in zip(raw,onsets): trials.append(x[:,o:o+2000])
    x=np.asarray(trials)*1e-6
    x=signal.resample_poly(x, 1, 2, axis=-1)
    sos=signal.butter(4, CONFIG["filter_band_hz"], btype="bandpass", fs=CONFIG["resample_sfreq"], output="sos")
    x=signal.sosfiltfilt(sos,x,axis=-1)
    if CONFIG["reference_mode"] == "average": x=x-x.mean(1,keepdims=True)
    elif CONFIG["reference_mode"] != "none": raise ValueError("reference_mode must be average or none")
    return x
def gfp_peaks(raw):
    kwargs={"picks":"eeg", "min_peak_distance":max(1,round(CONFIG["gfp_min_peak_distance_ms"]*CONFIG["resample_sfreq"]/1000))}
    if "return_all" in inspect.signature(extract_gfp_peaks).parameters: kwargs["return_all"]=False
    peaks=extract_gfp_peaks(raw, **kwargs)
    # pycrostates has no max_peaks parameter in some releases; deterministic subsampling enforces it.
    n=getattr(peaks,"n_times",0); cap=CONFIG["gfp_max_peaks"]
    return peaks if not cap or n <= cap else peaks.copy().crop(tmax=(cap-1)/peaks.info["sfreq"])
def temporal_features(x, centers, names):
    flat=x.transpose(1,0,2).reshape(x.shape[1],-1); flat-=flat.mean(0,keepdims=True)
    corr=np.abs(centers @ flat / ((np.linalg.norm(centers,axis=1)[:,None]+1e-12)*(np.linalg.norm(flat,axis=0)[None,:]+1e-12)))
    labels=corr.argmax(0); feats={}
    for k,name in enumerate(names):
        mask=labels==k; starts=np.flatnonzero(mask & np.r_[True,~mask[:-1]]); ends=np.flatnonzero(mask & np.r_[~mask[1:],True])
        feats[f"{name}_duration"]=float(np.mean(ends-starts+1)/CONFIG["resample_sfreq"] if len(starts) else 0)
        feats[f"{name}_occurrence"]=float(len(starts)/(len(labels)/CONFIG["resample_sfreq"])); feats[f"{name}_coverage"]=float(mask.mean())
    for a,an in enumerate(names):
        den=max(1,np.sum(labels[:-1]==a))
        for b,bn in enumerate(names): feats[f"{an}_to_{bn}"]=float(np.sum((labels[:-1]==a)&(labels[1:]==b))/den)
    return feats
def select_features(X,y,groups):
    names=list(X.columns); p=[]
    for n in names:
        d=[]
        for g in np.unique(groups):
            z=X.loc[groups==g,n].to_numpy(); yy=y[groups==g]
            if set(yy)=={0,1}: d.append(z[yy==1][0]-z[yy==0][0])
        p.append(stats.ttest_1samp(d,0).pvalue if len(d)>1 else 1.0)
    p=np.asarray(p); order=np.argsort(p); ranked=p[order]; qrank=np.minimum.accumulate((ranked*len(p)/np.arange(1,len(p)+1))[::-1])[::-1]; q=np.empty_like(qrank); q[order]=np.minimum(qrank,1.0); reject=q<=CONFIG["fdr_alpha"]
    significant=[n for n,r in zip(names,reject) if r]
    if significant: return significant,"fdr_significant",dict(zip(names,q))
    ranked=[n for _,n in sorted(zip(p,names))[:CONFIG["fallback_feature_count"]]]
    return ranked,"exploratory_fallback_not_significant",dict(zip(names,q))

# 4. Model
## 4.1 Transductive Group Maps and Features

# 5. Training
## 5.1 Grouped Evaluation
Feature selection is repeated using outer-training subjects only. FDR discoveries and exploratory fallback are separately labeled; fallback is never called significant.

In [ ]:
if CONFIG["evaluation_mode"] == "honest_grouped" and not CONFIG["allow_transductive_group_maps"]:
    raise RuntimeError("Group maps are fit once using all subjects. Set allow_transductive_group_maps=True or implement fold-local maps.")
rows=[]; inventory=[]; templates=[]; DATA={}
for p in selected_files():
    sid,raw,y,on=load_subject(p); x=preprocess(raw,on); qc=float(x.std(axis=(0,2)).mean()*1e6); flagged=qc>CONFIG["qc_max_mean_eeg_std_uv"]
    inventory.append({"subject_id":sid,"qc_mean_std_uv":qc,"qc_flagged":flagged,"excluded":bool(flagged and CONFIG["exclude_qc_failures"])})
    if flagged and CONFIG["exclude_qc_failures"]: continue
    DATA[sid]=(x,y)
    info=mne.create_info(CH_NAMES,CONFIG["resample_sfreq"],["eeg"]*len(CH_NAMES)); rr=mne.io.RawArray(x.transpose(1,0,2).reshape(len(CH_NAMES),-1),info,verbose=False)
    km=ModKMeans(CONFIG["n_states"],n_init=CONFIG["kmeans_n_init"],random_state=BASE_SEED); km.fit(gfp_peaks(rr),verbose=False); templates.extend(km.cluster_centers_)
pd.DataFrame(inventory).to_csv(ARTIFACT_DIR/"subject_inventory.csv",index=False)
meta=ModKMeans(CONFIG["n_states"],n_init=CONFIG["kmeans_n_init"],random_state=BASE_SEED)
from pycrostates.io import ChData
meta.fit(ChData(np.asarray(templates),mne.create_info(CH_NAMES,CONFIG["resample_sfreq"],["eeg"]*len(CH_NAMES))),verbose=False)
state_names=[f"MS{i}" for i in range(CONFIG["n_states"])]
if CONFIG["paper_state_label_map"]: state_names=[CONFIG["paper_state_label_map"].get(n,n) for n in state_names]
for sid,(x,y) in DATA.items():
    for label in [0,1]: rows.append({"subject_id":sid,"label":label,**temporal_features(x[y==label],meta.cluster_centers_,state_names)})
df=pd.DataFrame(rows); SUBJECTS=sorted(DATA); y=df.pop("label").to_numpy(); groups=df.pop("subject_id").to_numpy(); X=df
paper=[f for f in CONFIG["paper_reported_features"] if f in X.columns]
if CONFIG["evaluation_mode"]=="paper_style":
    splits=StratifiedKFold(10,shuffle=True,random_state=BASE_SEED).split(X,y); fixed=paper if len(paper)==9 else None
else: splits=LeaveOneGroupOut().split(X,y,groups)
FOLD_RESULTS=[]
for fold,(tr,te) in enumerate(splits):
    features,status,q= (fixed,"paper_reported_comparator",{}) if CONFIG["evaluation_mode"]=="paper_style" and fixed else select_features(X.iloc[tr],y[tr],groups[tr])
    for name in CONFIG["classifiers"]:
        clf={"lda":LinearDiscriminantAnalysis(),"svm":SVC(kernel="linear"),"knn":KNeighborsClassifier(min(CONFIG["knn_n_neighbors"],len(tr)-1))}[name]
        model=make_pipeline(StandardScaler(),clf).fit(X.iloc[tr][features],y[tr]); pred=model.predict(X.iloc[te][features])
        score=model.decision_function(X.iloc[te][features]) if name=="svm" else (model.predict_proba(X.iloc[te][features])[:,1] if hasattr(model,"predict_proba") else pred)
        FOLD_RESULTS.append({"fold_id":fold,"classifier":name,"test_subjects":groups[te].tolist(),"y_true":y[te].tolist(),"y_pred":pred.tolist(),"score":np.asarray(score).tolist(),"selected_features":features,"feature_status":status,"transductive_group_maps":True})
SUBJECT_METRICS=[]
for name in CONFIG["classifiers"]:
    rr=[r for r in FOLD_RESULTS if r["classifier"]==name]; yt=np.concatenate([r["y_true"] for r in rr]); yp=np.concatenate([r["y_pred"] for r in rr]); sc=np.concatenate([np.atleast_1d(r["score"]) for r in rr])
    SUBJECT_METRICS.append({"classifier":name,"pooled_oof_accuracy":accuracy_score(yt,yp),"pooled_oof_balanced_accuracy":balanced_accuracy_score(yt,yp),"pooled_oof_auc":roc_auc_score(yt,sc),"confusion_matrix":confusion_matrix(yt,yp).tolist()})
GLOBAL_METRICS={"primary":"pooled_out_of_fold","evaluation_mode":CONFIG["evaluation_mode"],"transductive_group_maps":True,"classifiers":SUBJECT_METRICS}
# Exact paired McNemar for each classifier pair; no uncorrected chi-square shortcut.
from scipy.stats import binomtest
GLOBAL_METRICS["mcnemar_exact"]={}
preds={r["classifier"]:np.concatenate([x["y_pred"] for x in FOLD_RESULTS if x["classifier"]==r["classifier"]]) for r in FOLD_RESULTS}
yt=np.concatenate([r["y_true"] for r in FOLD_RESULTS if r["classifier"]==CONFIG["classifiers"][0]])
for i,a in enumerate(CONFIG["classifiers"]):
    for b in CONFIG["classifiers"][i+1:]:
        n01=int(np.sum((preds[a]!=yt)&(preds[b]==yt))); n10=int(np.sum((preds[a]==yt)&(preds[b]!=yt)))
        GLOBAL_METRICS["mcnemar_exact"][f"{a}_vs_{b}"]={"n01":n01,"n10":n10,"pvalue":binomtest(min(n01,n10),n01+n10,0.5).pvalue if n01+n10 else 1.0}

# 6. Results
## 6.1 Aggregate and Save Artifacts

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
pd.DataFrame(FOLD_RESULTS).to_json(cv_results_path, orient="records", indent=2)
pd.DataFrame(SUBJECT_METRICS).to_json(subject_metrics_path, orient="records", indent=2)
with open(global_metrics_path, "w") as f: json.dump(GLOBAL_METRICS, f, indent=2)
run_metadata = {"run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR), "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"], "subjects": [int(s) for s in SUBJECTS], "channel_names": CH_NAMES, "seed": BASE_SEED, "global_metrics": GLOBAL_METRICS, "performance_artifacts": {"cv_results": str(cv_results_path), "subject_metrics": str(subject_metrics_path), "global_metrics": str(global_metrics_path)}}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f: json.dump(run_metadata, f, indent=2)
print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass